In [9]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"


# Update memory
@tool
def save_user_info(user_id: str, name: str, age: int, email: str, runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."


agent = create_agent(
    "deepseek-flash",
    tools=[get_user_info, save_user_info],
    store=InMemoryStore()
)

# First session: save user info
result = agent.invoke({
    "messages": [{"role": "user",
                  "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})
[message.content for message in result["messages"]]

['Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev',
 "I'll save that user for you.",
 'Successfully saved user info.',
 'Done! The user has been saved successfully:\n\n| Field | Value |\n|-------|-------|\n| User ID | `abc123` |\n| Name | Foo |\n| Age | 25 |\n| Email | foo@langchain.dev |']

In [12]:
# Second session: get user info
result = agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id abc123"}]
})
[message.content for message in result["messages"]]

['Get user info for user with id abc123',
 "I'll look up the user information for that ID.",
 "{'name': 'Foo', 'age': 25, 'email': 'foo@langchain.dev'}",
 "Here's the information for user `abc123`:\n\n| Field | Value |\n|-------|-------|\n| **Name** | Foo |\n| **Age** | 25 |\n| **Email** | foo@langchain.dev |\n\nIs there anything else you'd like me to do with this user's info?"]